In [1]:
import dotenv
import sqlite3
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt

dotenv.load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {"configurable": {"thread_id": "1"}}

In [2]:
class State(MessagesState):
    pass


graph_builder = StateGraph(State)

In [3]:
@tool
def get_human_feedback(poem: str):
    """Asks the user for feedback on the poem. Use this before returning the final response."""
    feedback = interrupt(f"Here is the poem, tell me what you think\n\n{poem}")
    return feedback


llm_with_tools = llm.bind_tools(tools=[get_human_feedback])


def chatbot(state: State):
    response = llm_with_tools.invoke(f"""
        You are an expert in making poems.
        Use the `get_human_feedback` tool to get feedback on your poem.
        Only after you receive positive feedback you can return the final poem.
        ALWAYS ASK FOR FEEDBACK FIRST.
        Here is the conversation history:
        {state["messages"]}
    """)
    return {"messages": [response]}

In [4]:
tool_node = ToolNode(tools=[get_human_feedback])

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [5]:
result = graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "Please make a poem about Python code."},
        ],
    },
    config=config,
)

In [6]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_dblGddI8NQEWWdCoRyKAk5tS)
 Call ID: call_dblGddI8NQEWWdCoRyKAk5tS
  Args:
    poem: In realms where logic intertwines,
A dance of code, where Python shines.
With syntax sleek and elegance profound,
In lines of script, solutions abound.

Indentations guide the way,
Through loops and functions, night and day.
A canvas where ideas take flight,
Crafting wonders in digital light.

Variables whisper, 'Store me, please!'
While algorithms hum with ease.
From data frames to queries bright,
Python’s magic ignites the night.

So here’s to the code, so lively, free,
An artist’s tool for creativity.
In every script, a journey unfolds,
With Python, the future surely beholds.


In [7]:
snapshot = graph.get_state(config)
snapshot.next

('tools',)

In [8]:
response = Command(resume="It looks great!")
result = graph.invoke(response, config=config)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_dblGddI8NQEWWdCoRyKAk5tS)
 Call ID: call_dblGddI8NQEWWdCoRyKAk5tS
  Args:
    poem: In realms where logic intertwines,
A dance of code, where Python shines.
With syntax sleek and elegance profound,
In lines of script, solutions abound.

Indentations guide the way,
Through loops and functions, night and day.
A canvas where ideas take flight,
Crafting wonders in digital light.

Variables whisper, 'Store me, please!'
While algorithms hum with ease.
From data frames to queries bright,
Python’s magic ignites the night.

So here’s to the code, so lively, free,
An artist’s tool for creativity.
In every script, a journey unfolds,
With Python, the future surely beholds.
================================= Tool Message =================================
Name: get

In [9]:
snapshot = graph.get_state(config)
snapshot.next

()